# 3-bit vs 4-bit experts — the perplexity A/B, on Colab

Answers the one question `docs/QUALITY.md` leaves open: **did quantizing
Qwen3.6-35B-A3B's experts to 3 bits cost anything?**

It has not been run on the Mac because the pair needs ~20 GB resident and an
attempt hit 23.3 GB of 23.5 GB swap in uninterruptible wait.

**Nothing is uploaded.** The 4-bit build is public on HF and Colab pulls it at
datacenter speed; the 3-bit build is *regenerated here* by the repo's own
`scripts/convert-q3-experts.py`, which is the same script that produced the
local one. The user's connection is never in the path.

This works at all because MLX shipped a CUDA backend (`pip install "mlx[cuda]"`),
so `mlx_lm` runs on an NVIDIA GPU. That backend is young and **not every
operator is implemented** — a 256-expert quantized MoE leans on `gather_qmm`,
which is exactly the kind of specialised op that might be missing. So cell 3
probes it in seconds, before anything downloads 19.7 GB.

## Runtime you need

**A100 40 GB, or any high-RAM runtime.** Free-tier T4 (16 GB) will not hold the
4-bit build. The A/B loads one model at a time (`del model; mx.clear_cache()`
between them), so peak is one checkpoint: ~19.7 GB.

Runtime → Change runtime type → A100 GPU + High-RAM.


## 1. What did we get?


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import os, psutil
print(f"system RAM {psutil.virtual_memory().total/2**30:.1f} GB, disk free "
      f"{psutil.disk_usage('/content').free/2**30:.1f} GB")
# Need ~40 GB disk: 19.7 GB for the 4-bit + ~15.7 GB for the 3-bit we build.


## 2. Install MLX with the CUDA backend


In [ ]:
!pip install -q "mlx[cuda]" mlx-lm
import mlx.core as mx
print('mlx', mx.__version__, '| default device', mx.default_device())


## 3. PROBE FIRST — does a quantized MoE actually run on this backend?

Seconds, no download. If `gather_qmm` is missing on CUDA the whole notebook is
dead and it is far better to learn that now than after 19.7 GB.


In [ ]:
import mlx.core as mx
ok = {}
try:
    w = mx.random.normal((256, 512))
    q, s, b = mx.quantize(w, group_size=64, bits=4)
    x = mx.random.normal((4, 512))
    mx.eval(mx.quantized_matmul(x, q, s, b, transpose=True, group_size=64, bits=4))
    ok['quantized_matmul 4-bit'] = True
except Exception as e:
    ok['quantized_matmul 4-bit'] = f'FAILED: {e}'

try:
    q3, s3, b3 = mx.quantize(mx.random.normal((256, 512)), group_size=64, bits=3)
    mx.eval(mx.quantized_matmul(mx.random.normal((4, 512)), q3, s3, b3,
                                transpose=True, group_size=64, bits=3))
    ok['quantized_matmul 3-bit'] = True
except Exception as e:
    ok['quantized_matmul 3-bit'] = f'FAILED: {e}'

# The MoE path: one quantized matmul per expert, gathered by index.
try:
    E = 8
    we = mx.random.normal((E, 256, 512))
    qe, se, be = mx.quantize(we, group_size=64, bits=3)
    xi = mx.random.normal((1, 4, 512))
    idx = mx.array([[0, 3, 5, 7]])
    mx.eval(mx.gather_qmm(xi, qe, se, be, rhs_indices=idx, transpose=True,
                          group_size=64, bits=3))
    ok['gather_qmm 3-bit (the MoE op)'] = True
except Exception as e:
    ok['gather_qmm 3-bit (the MoE op)'] = f'FAILED: {e}'

for k, v in ok.items():
    print(f'{"PASS" if v is True else "FAIL"}  {k}' + ('' if v is True else f'\n      {v}'))
if not all(v is True for v in ok.values()):
    print('\nSTOP. The CUDA backend cannot run this model. Do not download anything.')
    print('Fall back to a quiet Mac, or an A100 box with MLX built from source.')


## 4. The repo (for its converter and the A/B script)


In [ ]:
%cd /content
![ -d zero-tvm ] || git clone --depth 1 https://github.com/abgnydn/zero-tvm.git
%cd /content/zero-tvm
!git log --oneline -1


## 5. Pull the 4-bit build — 19.7 GB, minutes on Colab


In [ ]:
!pip install -q huggingface_hub hf_transfer
import os; os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'
!hf download lmstudio-community/Qwen3.6-35B-A3B-MLX-4bit \
    --local-dir /content/q36-4bit
!du -sh /content/q36-4bit


## 6. Regenerate the 3-bit build with the repo's own converter

`scripts/convert-q3-experts.py` is the script that produced the local
`qwen36q3` checkpoint. Same code, same group size, same expert selection —
so what comes out here is the build that ships, not an approximation of it.


In [ ]:
import re, pathlib
src = pathlib.Path('scripts/convert-q3-experts.py').read_text()
# The script hardcodes the author's local paths; point it at Colab's.
src = re.sub(r'^SRC\s*=.*$', 'SRC = "/content/q36-4bit"', src, flags=re.M)
src = re.sub(r'^DST\s*=.*$', 'DST = "/content/q36-q3exp"', src, flags=re.M)
pathlib.Path('/content/convert_colab.py').write_text(src)
print(re.search(r'^SRC.*$|^DST.*$', src, re.M).group(0))


In [ ]:
!python /content/convert_colab.py
!du -sh /content/q36-q3exp


## 7. The A/B

Identical windows for both checkpoints, error bars, a z score and a verdict.
bf16 — both sides get the same treatment and only the difference is read, so
f32 would double memory for nothing.

The corpus defaults to this repo's own markdown and TypeScript, which is
in-domain for the agentic-coding target and needs no dataset download.


In [ ]:
!cd /content/zero-tvm && python scripts/quality-ab.py \
    --a /content/q36-4bit \
    --b /content/q36-q3exp \
    --windows 24 --window 512 \
    --out /content/ab-qwen36.json


## 8. Bring back the result

A few KB of JSON, not a checkpoint. Paste it into `BENCH.md`'s
"3-bit experts" section, which currently says the number is unrun.


In [ ]:
import json
r = json.load(open('/content/ab-qwen36.json'))
print(json.dumps({k: v for k, v in r.items() if k != 'nll'}, indent=1))
from google.colab import files; files.download('/content/ab-qwen36.json')


---

## Reading the answer

| delta | verdict |
|---|---|
| z < 2 | not distinguishable — raise `--windows` before concluding anything |
| ≤ +10% | ship: the 3-bit build costs little |
| +10–25% | real. A task benchmark decides, not this |
| > +25% | the 3-bit build is materially worse; `qwen36q3` should not be the default |

**Perplexity understates reasoning damage.** Published work puts the
math-accuracy drop at 3-bit around 3× the perplexity drop, so a pass here is
necessary and not sufficient. `mlx_lm.evaluate` implements the lm-eval API and
is the next step if this comes back marginal.

**One caveat on the backend.** These numbers come from MLX-on-CUDA, and the
shipped model runs MLX-on-Metal — and the browser engine runs neither. That is
fine for a *comparison*: both sides run on the same backend, and only their
ratio is read. Do not quote the absolute perplexities as the Mac's numbers.
